# exp054 Stage 1 v2 — Pretrain (Colab Blackwell)

**Purpose**: Multi-source pretrain (XC + BC + iNat + AnuraSet) for backbone robustness

**Inputs** (6 Kaggle Datasets):
- XC Part 1 (76 Aves)
- XC Part 2 (83 Aves)
- XC Part 3 (27 non-Aves)
- BC extracted (54 species, 過去 BC 2021-2025)
- iNat extracted (183 species)
- AnuraSet extracted (17 Amphibia)

**Coverage**: ~210-220 / 234 unique BC2026 species

**Model**: `tf_efficientnetv2_s.in21k_ft_in1k` (paper 256 採用)
**Train**: 25 epochs, batch 192, lr 1.2e-3, bf16 AMP, MIXUP α=1.5, drop_path 0.2
**Expected**: 30-60 min on Colab Blackwell (RTX PRO 6000)

**Output**: backbone state_dict → Kaggle Dataset `maekeso/birdclef2026-exp054-stage1-effv2s`

vs Stage 1 (Phase 1):
- Phase 1: XC only (~26k files, 159 Aves species) → val_ns22 0.9423
- Phase 2: 6 sources (~46k files, 210+ species) → expected val_ns22 0.94+ + better non-Aves


In [ ]:
# Cell 1: install deps + drive mount
!pip install -q timm==1.0.11 soundfile librosa kaggle 2>&1 | tail -1

from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Drive root (per memory: kaggle/birdclef2026/)
DRIVE_ROOT = Path("/content/drive/MyDrive/kaggle/birdclef2026/output/exp054")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Drive: {DRIVE_ROOT}")


In [ ]:
# Cell 2: Download 6 Kaggle Datasets to local (SDK + KAGGLE_API_TOKEN env var)
import os, json, zipfile, time
from pathlib import Path

# Setup kaggle.json (used as source of KGAT_ token)
KAGGLE_DIR = Path.home() / ".kaggle"
KAGGLE_DIR.mkdir(exist_ok=True)
if not (KAGGLE_DIR / "kaggle.json").exists():
    src_kg = Path("/content/drive/MyDrive/kaggle/kaggle.json")
    if src_kg.exists():
        import shutil; shutil.copy(src_kg, KAGGLE_DIR / "kaggle.json")
        os.chmod(KAGGLE_DIR / "kaggle.json", 0o600)
        print("  ✓ Copied kaggle.json from Drive")
    else:
        raise SystemExit("Need kaggle.json at /content/drive/MyDrive/kaggle/kaggle.json")

# CRITICAL: set KAGGLE_API_TOKEN env var (KGAT_ Bearer token)
# CLI uses Basic auth (username/key) which fails with Bearer tokens.
# SDK + KAGGLE_API_TOKEN env var = Bearer auth (working).
_kgat = json.loads((KAGGLE_DIR / "kaggle.json").read_text())["key"]
os.environ["KAGGLE_API_TOKEN"] = _kgat
print(f"  KAGGLE_API_TOKEN set (len={len(_kgat)}, starts with {_kgat[:5]}...)")

from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()

DATASETS = [
    "maekeso/birdclef2026-xc-api-dl-part1",
    "maekeso/birdclef2026-xc-api-dl-part2",
    "maekeso/birdclef2026-xc-api-dl-part3",
    "maekeso/birdclef2026-exp047-bc-extracted",
    "maekeso/birdclef2026-exp047-inat-extracted",
    "maekeso/birdclef2026-exp047-anuraset-extracted",
]

LOCAL_DATA = Path("/content/data")
LOCAL_DATA.mkdir(exist_ok=True)

t0 = time.time()
for ds in DATASETS:
    name = ds.split("/")[-1]
    dst = LOCAL_DATA / name
    if dst.exists() and any(dst.iterdir()):
        n_files = sum(1 for _ in dst.rglob("*") if _.is_file())
        if n_files > 0:
            print(f"  ✓ {name} exists ({n_files} files), skip")
            continue
    dst.mkdir(exist_ok=True)
    print(f"  DL {ds}...")
    try:
        api.dataset_download_files(ds, path=str(dst), unzip=True, quiet=False)
        n_files = sum(1 for _ in dst.rglob("*") if _.is_file())
        gb = sum(f.stat().st_size for f in dst.rglob("*") if f.is_file()) / 1e9
        print(f"    ✓ {n_files} files, {gb:.2f} GB")
    except Exception as e:
        print(f"    ✗ err: {str(e)[:200]}")

print(f"\nDL total: {(time.time()-t0)/60:.1f} min")

# Final verify
print(f"\n=== Final verify ===")
total_files = 0; total_gb = 0
for ds in DATASETS:
    name = ds.split("/")[-1]
    dst = LOCAL_DATA / name
    if dst.exists():
        n_files = sum(1 for _ in dst.rglob("*") if _.is_file())
        gb = sum(f.stat().st_size for f in dst.rglob("*") if f.is_file()) / 1e9
        total_files += n_files; total_gb += gb
        print(f"  {name}: {n_files} files, {gb:.2f} GB")
    else:
        print(f"  ✗ {name}: missing")
print(f"\nTOTAL: {total_files} files, {total_gb:.2f} GB")
# Soft assertion: warn if any source missing, but continue
missing = [ds.split("/")[-1] for ds in DATASETS
           if not (LOCAL_DATA / ds.split("/")[-1]).exists()
           or sum(1 for _ in (LOCAL_DATA / ds.split("/")[-1]).rglob("*") if _.is_file()) == 0]
if missing:
    print(f"\n⚠ MISSING sources: {missing}")
    print(f"  Continuing with available data only ({total_files} files)")
assert total_files > 5000, f"Too few files ({total_files}) - check Kaggle auth"


In [ ]:
# Cell 3: imports + GPU info
import os, sys, json, time, math, random, gc, re, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
# Use new torch.amp API (torch.cuda.amp is deprecated in PyTorch 2.x)
from torch.amp import autocast, GradScaler
import torchaudio
import soundfile as sf
import librosa
import timm
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("high")
print(f"Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")
print(f"GPU mem: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"torch: {torch.__version__}, timm: {timm.__version__}")


In [ ]:
# Cell 4: BC2026 species list (234)
__BC2026_SPECIES = [
    ('Guyalna cuta', '1161364', 'Insecta', 1161364),
    ('Caiman yacare', '116570', 'Reptilia', 116570),
    ('Leptodactylus luctator', '1176823', 'Amphibia', 1176823),
    ('Adenomera guarani', '1491113', 'Amphibia', 1491113),
    ('Lysapsus limellum', '1595929', 'Amphibia', 1595929),
    ('Equus caballus', '209233', 'Mammalia', 209233),
    ('Leptodactylus syphax', '22930', 'Amphibia', 22930),
    ('Leptodactylus mystacinus', '22956', 'Amphibia', 22956),
    ('Leptodactylus podicipinus', '22961', 'Amphibia', 22961),
    ('Leptodactylus elenae', '22967', 'Amphibia', 22967),
    ('Leptodactylus fuscus', '22973', 'Amphibia', 22973),
    ('Leptodactylus labyrinthicus', '22983', 'Amphibia', 22983),
    ('Leptodactylus petersii', '22985', 'Amphibia', 22985),
    ('Physalaemus centralis', '23150', 'Amphibia', 23150),
    ('Physalaemus albifrons', '23154', 'Amphibia', 23154),
    ('Physalaemus albonotatus', '23158', 'Amphibia', 23158),
    ('Pseudopaludicola mystacalis', '23176', 'Amphibia', 23176),
    ('Phyllomedusa sauvagii', '23724', 'Amphibia', 23724),
    ('Scinax nasicus', '24279', 'Amphibia', 24279),
    ('Scinax fuscovarius', '24285', 'Amphibia', 24285),
    ('Scinax fuscomarginatus', '24287', 'Amphibia', 24287),
    ('Scinax acuminatus', '24321', 'Amphibia', 24321),
    ('Quesada gigas', '244024', 'Insecta', 244024),
    ('Chiasmocleis mehelyi', '25073', 'Amphibia', 25073),
    ('Elachistocleis bicolor', '25092', 'Amphibia', 25092),
    ('Dermatonotus muelleri', '25214', 'Amphibia', 25214),
    ('Physalaemus biligonigerus', '326272', 'Amphibia', 326272),
    ('Panthera onca', '41970', 'Mammalia', 41970),
    ('Alouatta caraya', '43435', 'Mammalia', 43435),
    ('Canis familiaris', '47144', 'Mammalia', 47144),
    ('Insect son01', '47158son01', 'Insecta', 47158),
    ('Insect son02', '47158son02', 'Insecta', 47158),
    ('Insect son03', '47158son03', 'Insecta', 47158),
    ('Insect son04', '47158son04', 'Insecta', 47158),
    ('Insect son05', '47158son05', 'Insecta', 47158),
    ('Insect son06', '47158son06', 'Insecta', 47158),
    ('Insect son07', '47158son07', 'Insecta', 47158),
    ('Insect son08', '47158son08', 'Insecta', 47158),
    ('Insect son09', '47158son09', 'Insecta', 47158),
    ('Insect son10', '47158son10', 'Insecta', 47158),
    ('Insect son11', '47158son11', 'Insecta', 47158),
    ('Insect son12', '47158son12', 'Insecta', 47158),
    ('Insect son13', '47158son13', 'Insecta', 47158),
    ('Insect son14', '47158son14', 'Insecta', 47158),
    ('Insect son15', '47158son15', 'Insecta', 47158),
    ('Insect son16', '47158son16', 'Insecta', 47158),
    ('Insect son17', '47158son17', 'Insecta', 47158),
    ('Insect son18', '47158son18', 'Insecta', 47158),
    ('Insect son19', '47158son19', 'Insecta', 47158),
    ('Insect son20', '47158son20', 'Insecta', 47158),
    ('Insect son21', '47158son21', 'Insecta', 47158),
    ('Insect son22', '47158son22', 'Insecta', 47158),
    ('Insect son23', '47158son23', 'Insecta', 47158),
    ('Insect son24', '47158son24', 'Insecta', 47158),
    ('Insect son25', '47158son25', 'Insecta', 47158),
    ('Physalaemus nattereri', '476521', 'Amphibia', 476521),
    ('Sapajus cay', '516975', 'Mammalia', 516975),
    ('Pithecopus azureus', '517063', 'Amphibia', 517063),
    ('Boana lundii', '555123', 'Amphibia', 555123),
    ('Boana punctata', '555145', 'Amphibia', 555145),
    ('Boana raniceps', '555146', 'Amphibia', 555146),
    ('Ameerega picta', '64898', 'Amphibia', 64898),
    ('Dendropsophus minutus', '65377', 'Amphibia', 65377),
    ('Dendropsophus nanus', '65380', 'Amphibia', 65380),
    ('Pseudis platensis', '66971', 'Amphibia', 66971),
    ('Rhinella diptycha', '67107', 'Amphibia', 67107),
    ('Trachycephalus typhonius', '67252', 'Amphibia', 67252),
    ('Leptodactylus macrosternum', '70711', 'Amphibia', 70711),
    ('Plecturocebus pallescens', '738183', 'Mammalia', 738183),
    ('Bos taurus', '74113', 'Mammalia', 74113),
    ('Mico melanurus', '74580', 'Mammalia', 74580),
    ('Prionacris erosa', '760266', 'Insecta', 760266),
    ('Hylophilus pectoralis', 'ashgre1', 'Aves', 17431),
    ('Mustelirallus albicollis', 'astcra1', 'Aves', 508907),
    ('Crax fasciolata', 'bafcur1', 'Aves', 2046),
    ('Micrastur ruficollis', 'baffal1', 'Aves', 4699),
    ('Coereba flaveola', 'banana', 'Aves', 10199),
    ('Thamnophilus doliatus', 'barant1', 'Aves', 15764),
    ('Procnias nudicollis', 'batbel1', 'Aves', 8854),
    ('Ara ararauna', 'baymac', 'Aves', 19018),
    ('Dendrocygna autumnalis', 'bbwduc', 'Aves', 6893),
    ('Microspingus melanoleucus', 'bcwfin2', 'Aves', 558564),
    ('Donacobius atricapilla', 'bkcdon', 'Aves', 116877),
    ('Aratinga nenday', 'bkhpar', 'Aves', 367562),
    ('Busarellus nigricollis', 'blchaw1', 'Aves', 5346),
    ('Spizaetus tyrannus', 'blheag1', 'Aves', 5291),
    ('Tityra cayana', 'blttit1', 'Aves', 8830),
    ('Myiarchus tyrannulus', 'bncfly', 'Aves', 16016),
    ('Megarynchus pitangua', 'bobfly1', 'Aves', 16737),
    ('Progne tapera', 'brcmar1', 'Aves', 11870),
    ('Tyto furcata', 'brnowl', 'Aves', 1578502),
    ('Momotus momota', 'bucmot4', 'Aves', 204447),
    ('Thectocercus acuticaudatus', 'bucpar', 'Aves', 367564),
    ('Amazona aestiva', 'bufpar', 'Aves', 18978),
    ('Theristicus caudatus', 'bunibi1', 'Aves', 3766),
    ('Athene cunicularia', 'burowl', 'Aves', 19975),
    ('Colaptes campestris', 'camfli1', 'Aves', 18262),
    ('Ortalis canicollis', 'chacha1', 'Aves', 2088),
    ('Mimus saturninus', 'chbmoc1', 'Aves', 14878),
    ('Gnorimopsar chopi', 'chobla1', 'Aves', 10723),
    ('Conirostrum speciosum', 'chvcon1', 'Aves', 10013),
    ('Synallaxis hypospodia', 'cibspi1', 'Aves', 10921),
    ('Micrastur semitorquatus', 'coffal1', 'Aves', 4698),
    ('Nyctidromus albicollis', 'compau', 'Aves', 19627),
    ('Nyctibius griseus', 'compot1', 'Aves', 19667),
    ('Turdus amaurochalinus', 'crbthr1', 'Aves', 12710),
    ('Pachyramphus validus', 'crebec1', 'Aves', 8681),
    ('Taoniscus nanus', 'dwatin1', 'Aves', 20714),
    ('Icterus pyrrhopterus', 'epaori4', 'Aves', 72954),
    ('Lathrotriccus euleri', 'eulfly1', 'Aves', 17231),
    ('Cantorchilus guarayanus', 'fabwre1', 'Aves', 144880),
    ('Glaucidium brasilianum', 'fepowl', 'Aves', 19822),
    ('Machaeropterus pyrocephalus', 'ficman1', 'Aves', 14344),
    ('Myiothlypis flaveola', 'flawar1', 'Aves', 201223),
    ('Tyrannus savana', 'fotfly', 'Aves', 16793),
    ('Cnemotriccus fuscatus', 'fusfly1', 'Aves', 17076),
    ('Hylocharis chrysura', 'gilhum1', 'Aves', 5979),
    ('Aramides ypecaha', 'giwrai1', 'Aves', 460),
    ('Chionomesa fimbriata', 'glteme1', 'Aves', 1289639),
    ('Saltator coerulescens', 'grasal3', 'Aves', 9850),
    ('Crotophaga major', 'greani1', 'Aves', 1970),
    ('Taraba major', 'greant1', 'Aves', 15957),
    ('Myiopagis viridicata', 'greela', 'Aves', 16892),
    ('Pitangus sulphuratus', 'grekis', 'Aves', 16956),
    ('Nyctibius grandis', 'grepot1', 'Aves', 19680),
    ('Phacellodomus ruber', 'gretho2', 'Aves', 11632),
    ('Tringa melanoleuca', 'greyel', 'Aves', 3892),
    ('Leptotila rufaxilla', 'grfdov1', 'Aves', 3302),
    ('Eucometis penicillata', 'grhtan1', 'Aves', 10698),
    ('Aramides cajaneus', 'gycwor1', 'Aves', 513889),
    ('Anhima cornuta', 'horscr1', 'Aves', 6908),
    ('Passer domesticus', 'houspa', 'Aves', 13858),
    ('Anodorhynchus hyacinthinus', 'hyamac1', 'Aves', 18938),
    ('Elaenia spectabilis', 'larela1', 'Aves', 16734),
    ('Elaenia chiriquensis', 'lesela1', 'Aves', 578460),
    ('Emberizoides ypiranganus', 'lesgrf1', 'Aves', 10555),
    ('Aramus guarauna', 'limpki', 'Aves', 7),
    ('Dryocopus lineatus', 'linwoo1', 'Aves', 17858),
    ('Coccycua minuta', 'litcuc2', 'Aves', 72740),
    ('Setopagis parvula', 'litnig1', 'Aves', 367507),
    ('Pyrrhura frontalis', 'mabpar', 'Aves', 19162),
    ('Cercomacra melanaria', 'magant1', 'Aves', 15737),
    ('Cissopis leverianus', 'magtan2', 'Aves', 72727),
    ('Polioptila dumicola', 'masgna1', 'Aves', 7509),
    ('Chordeiles nacunda', 'nacnig1', 'Aves', 19661),
    ('Rufirallus schomburgkii', 'ocecra1', 'Aves', 1506288),
    ('Sittasomus griseicapillus', 'oliwoo1', 'Aves', 11511),
    ('Icterus croconotus', 'orbtro3', 'Aves', 62564),
    ('Amazona amazonica', 'orwpar', 'Aves', 18982),
    ('Pandion haliaetus', 'osprey', 'Aves', 116999),
    ('Synallaxis albescens', 'pabspi1', 'Aves', 10999),
    ('Furnarius leucopus', 'palhor3', 'Aves', 11281),
    ('Thraupis palmarum', 'paltan1', 'Aves', 10297),
    ('Dromococcyx phasianellus', 'phecuc1', 'Aves', 1982),
    ('Patagioenas picazuro', 'picpig2', 'Aves', 3102),
    ('Legatus leucophaius', 'pirfly1', 'Aves', 17312),
    ('Thamnophilus pelzelni', 'plasla1', 'Aves', 73493),
    ('Inezia inornata', 'platyr1', 'Aves', 16344),
    ('Cyanocorax chrysops', 'plcjay1', 'Aves', 8484),
    ('Theristicus caerulescens', 'pluibi1', 'Aves', 3768),
    ('Cyanocorax cyanomelas', 'purjay1', 'Aves', 8483),
    ('Hemitriccus margaritaceiventer', 'pvttyr1', 'Aves', 16273),
    ('Ara chloropterus', 'ragmac1', 'Aves', 19016),
    ('Campylorhamphus trochilirostris', 'rebscy1', 'Aves', 11201),
    ('Coryphospingus cucullatus', 'recfin1', 'Aves', 10310),
    ('Gallus gallus', 'redjun', 'Aves', 882),
    ('Cariama cristata', 'relser1', 'Aves', 14),
    ('Megaceryle torquata', 'rinkin1', 'Aves', 2552),
    ('Myiothlypis rivularis', 'rivwar1', 'Aves', 145267),
    ('Rupornis magnirostris', 'roahaw', 'Aves', 201041),
    ('Turdus rufiventris', 'rubthr1', 'Aves', 12738),
    ('Pseudoseisura unirufa', 'rufcac2', 'Aves', 11718),
    ('Casiornis rufus', 'rufcas2', 'Aves', 17102),
    ('Conopophaga lineata', 'rufgna3', 'Aves', 578313),
    ('Furnarius rufus', 'rufhor2', 'Aves', 11275),
    ('Antrostomus rufus', 'rufnig1', 'Aves', 201066),
    ('Phacellodomus rufifrons', 'ruftho1', 'Aves', 11624),
    ('Poecilotriccus latirostris', 'ruftof1', 'Aves', 17026),
    ('Myiozetetes cayanensis', 'rumfly1', 'Aves', 16833),
    ('Tigrisoma lineatum', 'ruther1', 'Aves', 5048),
    ('Galbula ruficauda', 'rutjac1', 'Aves', 1468),
    ('Arremon flavirostris', 'sabspa1', 'Aves', 10064),
    ('Sicalis flaveola', 'saffin', 'Aves', 9864),
    ('Thraupis sayaca', 'saytan1', 'Aves', 10293),
    ('Columbina squammata', 'scadov1', 'Aves', 3564),
    ('Pionus maximiliani', 'schpar1', 'Aves', 19094),
    ('Phaethornis eurynome', 'scther1', 'Aves', 5622),
    ('Myiarchus ferox', 'shcfly1', 'Aves', 16006),
    ('Accipiter striatus', 'shshaw', 'Aves', 5097),
    ('Lurocalis semitorquatus', 'shtnig1', 'Aves', 19645),
    ('Ramphocelus carbo', 'sibtan2', 'Aves', 10056),
    ('Crotophaga ani', 'smbani', 'Aves', 1971),
    ('Crypturellus parvirostris', 'smbtin1', 'Aves', 20570),
    ('Cacicus solitarius', 'sobcac1', 'Aves', 10365),
    ('Camptostoma obsoletum', 'sobtyr1', 'Aves', 16972),
    ('Myiozetetes similis', 'socfly1', 'Aves', 16842),
    ('Synallaxis frontalis', 'sofspi1', 'Aves', 10996),
    ('Corythopis delalandi', 'souant1', 'Aves', 17264),
    ('Vanellus chilensis', 'soulap1', 'Aves', 4867),
    ('Chauna torquata', 'souscr1', 'Aves', 6910),
    ('Hypoedaleus guttatus', 'spbant3', 'Aves', 15959),
    ('Synallaxis spixi', 'spispi1', 'Aves', 10915),
    ('Antiurus maculicaudus', 'sptnig1', 'Aves', 1584760),
    ('Piaya cayana', 'squcuc1', 'Aves', 1758),
    ('Dendroplex picus', 'stbwoo2', 'Aves', 72806),
    ('Tapera naevia', 'strcuc1', 'Aves', 1989),
    ('Butorides striata', 'strher2', 'Aves', 62528),
    ('Asio clamator', 'strowl1', 'Aves', 558468),
    ('Eupetomena macroura', 'swthum1', 'Aves', 6065),
    ('Chiroxiphia caudata', 'swtman1', 'Aves', 14306),
    ('Crypturellus tataupa', 'tattin1', 'Aves', 20587),
    ('Campylorhynchus turdinus', 'thlwre1', 'Aves', 7480),
    ('Ramphastos toco', 'toctou1', 'Aves', 18793),
    ('Tyrannus melancholicus', 'trokin', 'Aves', 16787),
    ('Megascops choliba', 'trsowl', 'Aves', 19788),
    ('Crypturellus undulatus', 'undtin1', 'Aves', 20592),
    ('Thamnophilus caerulescens', 'varant1', 'Aves', 15757),
    ('Jacana jacana', 'watjac1', 'Aves', 4580),
    ('Pyriglena maura', 'wesfie1', 'Aves', 1286886),
    ('Dendrocygna viduata', 'wfwduc1', 'Aves', 6898),
    ('Biatas nigropectus', 'whbant2', 'Aves', 15902),
    ('Myiothlypis leucoblephara', 'whbwar2', 'Aves', 201224),
    ('Melanerpes candidus', 'whiwoo1', 'Aves', 18183),
    ('Synallaxis albilora', 'whlspi1', 'Aves', 10992),
    ('Cyanocorax cyanopogon', 'whnjay1', 'Aves', 8469),
    ('Leptotila verreauxi', 'whtdov', 'Aves', 3280),
    ('Picumnus albosquamatus', 'whwpic1', 'Aves', 17786),
    ('Caracara plancus', 'y00678', 'Aves', 4715),
    ('Paroaria capitata', 'yebcar', 'Aves', 10257),
    ('Elaenia flavogaster', 'yebela1', 'Aves', 16714),
    ('Primolius auricollis', 'yecmac', 'Aves', 73272),
    ('Brotogeris chiriri', 'yecpar', 'Aves', 19215),
    ('Daptrius chimachima', 'yehcar1', 'Aves', 1432779),
    ('Tolmomyias sulphurescens', 'yeofly1', 'Aves', 16567),
]
species_df = pd.DataFrame(__BC2026_SPECIES, columns=["scientific_name", "primary_label", "class_name", "inat_taxon_id"])
PRIMARY_LABELS = species_df["primary_label"].tolist()
N_CLASSES = len(PRIMARY_LABELS)
label_to_idx = {l: i for i, l in enumerate(PRIMARY_LABELS)}
SCI_LC_TO_LABEL = {str(s).lower(): l for s, l in zip(species_df['scientific_name'], species_df['primary_label'])}
LABEL_TO_CLASS = dict(zip(species_df['primary_label'], species_df['class_name']))
print(f"BC2026: {N_CLASSES} species")
print(f"  by class: {species_df['class_name'].value_counts().to_dict()}")


In [ ]:
# Cell 5: training config (paper 256 spec, adjusted for Blackwell)
CFG = dict(
    backbone="tf_efficientnetv2_s.in21k_ft_in1k",
    sr=32000, chunk_sec=5, n_mels=128, n_fft=2048, hop_length=512,
    fmin=20, fmax=16000,
    epochs=25,
    batch_size=192,
    lr=1.2e-3,                # sqrt(192/32) * 5e-4 ≈ 1.2e-3
    weight_decay=1e-3,
    warmup_steps=800,         # larger batch + higher lr → longer warmup
    label_smoothing=0.1,
    mixup_alpha=1.5,          # ↑ from 1.0 for stronger regularization
    grad_clip=2.0,
    num_workers=8,            # ↑ from 4 for batch 192 I/O
    val_split=0.1,
    val_seed=42,
)
for k, v in CFG.items(): print(f"  {k}: {v}")
CHUNK_LEN = CFG["sr"] * CFG["chunk_sec"]


In [ ]:
# Cell 6: build unified meta_df from all 6 sources
LOCAL_DATA = Path("/content/data")

all_records = []
total_skipped = 0

# Helper: extract primary_label from path or metadata
def from_xc_dataset(src_root, source_name):
    """XC: structure is xc_api[_partN]/<species_code>/<files>.mp3
       species_code can be in BC2026 or not (filter to BC2026)"""
    found = []
    # XC has .mp3 + .ogg + sometimes .wav, check all common audio extensions
    audio_files = []
    for ext in ["*.mp3", "*.ogg", "*.wav", "*.flac", "*.MP3", "*.OGG"]:
        audio_files.extend(list(src_root.rglob(ext)))
    for p in audio_files:
        sp_dir = p.parent.name
        if sp_dir in label_to_idx:
            found.append((str(p), sp_dir, source_name))
        else:
            sp_dir2 = p.parent.parent.name if p.parent.parent != src_root else None
            if sp_dir2 in label_to_idx:
                found.append((str(p), sp_dir2, source_name))
    return found

def from_extracted(src_root, source_name):
    """BC/iNat/AnuraSet extracted: metadata.csv has primary_label + filename"""
    meta_csv = next(src_root.rglob("metadata.csv"), None)
    if not meta_csv:
        return []
    df = pd.read_csv(meta_csv)
    found = []
    # filename in CSV is relative to extracted dir (e.g., "audio/bc2021/banana/XC112602.ogg")
    base = meta_csv.parent
    for _, r in df.iterrows():
        pl = str(r["primary_label"])
        if pl not in label_to_idx:
            continue
        rel = str(r["filename"])
        abs_path = base / rel
        if not abs_path.exists():
            # Try without "extracted/" prefix
            abs_path = base / rel.replace("extracted/", "")
        if abs_path.exists():
            found.append((str(abs_path), pl, source_name))
    return found

# Process each source
SOURCES_XC = [
    ("birdclef2026-xc-api-dl-part1", "xc1"),
    ("birdclef2026-xc-api-dl-part2", "xc2"),
    ("birdclef2026-xc-api-dl-part3", "xc3"),
]
SOURCES_EXTRACTED = [
    ("birdclef2026-exp047-bc-extracted", "bc"),
    ("birdclef2026-exp047-inat-extracted", "inat"),
    ("birdclef2026-exp047-anuraset-extracted", "anuraset"),
]

t0 = time.time()
for ds_name, src in SOURCES_XC:
    src_root = LOCAL_DATA / ds_name
    if not src_root.exists():
        print(f"⚠ {ds_name} not found, skip")
        continue
    records = from_xc_dataset(src_root, src)
    all_records.extend(records)
    print(f"  {src}: {len(records)} files")

for ds_name, src in SOURCES_EXTRACTED:
    src_root = LOCAL_DATA / ds_name
    if not src_root.exists():
        print(f"⚠ {ds_name} not found, skip")
        continue
    records = from_extracted(src_root, src)
    all_records.extend(records)
    print(f"  {src}: {len(records)} files")

meta_df = pd.DataFrame(all_records, columns=["filepath", "primary_label", "source"])
print(f"\nTotal: {len(meta_df)} files")
print(f"  by source: {meta_df['source'].value_counts().to_dict()}")
print(f"  unique species: {meta_df['primary_label'].nunique()} / {N_CLASSES}")
print(f"  build time: {(time.time()-t0)/60:.1f} min")

# Train/val split (stratified by species)
np.random.seed(CFG["val_seed"])
meta_df["target_idx"] = meta_df["primary_label"].map(label_to_idx)
meta_df = meta_df.sample(frac=1, random_state=CFG["val_seed"]).reset_index(drop=True)

val_mask = np.zeros(len(meta_df), dtype=bool)
# Per-species val sampling — min 2 per species for AUC computability
for sp in meta_df["primary_label"].unique():
    sp_idx = meta_df.index[meta_df["primary_label"] == sp].tolist()
    n_val = max(2, int(len(sp_idx) * CFG["val_split"]))  # min 2 for AUC (need 2+ positive)
    val_picks = np.random.choice(sp_idx, size=min(n_val, len(sp_idx)), replace=False)
    val_mask[val_picks] = True

train_df = meta_df[~val_mask].reset_index(drop=True)
val_df = meta_df[val_mask].reset_index(drop=True)
print(f"\nTrain: {len(train_df)} files, Val: {len(val_df)} files")
print(f"  val species: {val_df['primary_label'].nunique()}")


In [ ]:
# Cell 7: Dataset / DataLoader
class AudioDataset(Dataset):
    def __init__(self, df, sr, chunk_len, training=True):
        self.df = df.reset_index(drop=True)
        self.sr = sr
        self.chunk_len = chunk_len
        self.training = training

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row["filepath"]
        target = int(row["target_idx"])
        try:
            wav, sr = sf.read(path, dtype="float32")
            if wav.ndim > 1:
                wav = wav.mean(axis=1)
            if sr != self.sr:
                wav = librosa.resample(wav, orig_sr=sr, target_sr=self.sr)
        except Exception:
            wav = np.zeros(self.chunk_len, dtype=np.float32)

        if len(wav) < self.chunk_len:
            wav = np.pad(wav, (0, self.chunk_len - len(wav)))
        else:
            if self.training:
                start = np.random.randint(0, len(wav) - self.chunk_len + 1)
            else:
                start = (len(wav) - self.chunk_len) // 2
            wav = wav[start:start + self.chunk_len]

        return torch.from_numpy(wav).float(), target


In [ ]:
# Cell 8: Model definition (EffNetV2-S + SED head)
class MelExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=CFG["sr"], n_fft=CFG["n_fft"], hop_length=CFG["hop_length"],
            n_mels=CFG["n_mels"], f_min=CFG["fmin"], f_max=CFG["fmax"],
        )
        self.db = torchaudio.transforms.AmplitudeToDB(top_db=80.0)
    def forward(self, wav):
        mel = self.mel(wav)
        mel = self.db(mel)
        mel = torch.clamp(mel, -80.0, 0.0)
        mel = (mel + 40.0) / 40.0
        return mel

class SEDHead(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()
        self.att = nn.Linear(in_dim, n_classes)
        self.cla = nn.Linear(in_dim, n_classes)
    def forward(self, x):
        # x: (B, T, F)
        att = torch.tanh(self.att(x))
        cla = self.cla(x)
        norm_att = F.softmax(att, dim=1)
        clip = (norm_att * cla).sum(dim=1)
        return clip

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            CFG["backbone"], pretrained=True, in_chans=3,
            num_classes=0, global_pool="",
            drop_path_rate=0.2,       # ↑ Stochastic Depth for regularization
        )
        self.feat_dim = self.backbone.num_features
        self.head = SEDHead(self.feat_dim, N_CLASSES)
    def forward(self, mel):
        # mel: (B, F, T) -> (B, 3, F, T)
        x = mel.unsqueeze(1).repeat(1, 3, 1, 1)
        feat = self.backbone(x)  # (B, C, H', T')
        feat = feat.mean(dim=2).transpose(1, 2)  # (B, T', C)
        logit = self.head(feat)
        return logit


In [ ]:
# Cell 9: Augmentations (MixUp + SpecAugment)
def mixup_audio(x, y, alpha=1.0):
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    perm = torch.randperm(x.size(0), device=x.device)
    x2 = lam * x + (1 - lam) * x[perm]
    return x2, y, y[perm], lam

def spec_augment(mel, freq_mask=30, time_mask=60, n_freq=2, n_time=2):  # ↑ 20→30, 40→60
    B, F_, T = mel.shape
    for _ in range(n_freq):
        f = np.random.randint(0, freq_mask + 1)
        f0 = np.random.randint(0, max(1, F_ - f))
        mel[:, f0:f0+f, :] = 0
    for _ in range(n_time):
        t = np.random.randint(0, time_mask + 1)
        t0 = np.random.randint(0, max(1, T - t))
        mel[:, :, t0:t0+t] = 0
    return mel


In [ ]:
# Cell 10: DataLoader + optim setup
train_ds = AudioDataset(train_df, CFG["sr"], CHUNK_LEN, training=True)
val_ds = AudioDataset(val_df, CFG["sr"], CHUNK_LEN, training=False)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
                          num_workers=CFG["num_workers"], pin_memory=True, drop_last=True, persistent_workers=True)
val_loader = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False,
                        num_workers=CFG["num_workers"], pin_memory=True, persistent_workers=True)

n_steps = len(train_loader) * CFG["epochs"]
print(f"Steps/epoch: {len(train_loader)}, total: {n_steps}")


In [ ]:
# Cell 11: model + optimizer + scheduler
mel_extractor = MelExtractor().to(DEVICE)
model = Model().to(DEVICE)
print(f"Model params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

optimizer = optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])

def lr_lambda(step):
    if step < CFG["warmup_steps"]:
        return step / max(1, CFG["warmup_steps"])
    progress = (step - CFG["warmup_steps"]) / max(1, n_steps - CFG["warmup_steps"])
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


In [ ]:
# Cell 12: validation eval (matches exp052/053 log style)
@torch.no_grad()
def evaluate():
    model.eval()
    all_logits = []
    all_targets = []
    for wav, tgt in val_loader:
        wav = wav.to(DEVICE, non_blocking=True)
        with autocast('cuda', dtype=torch.bfloat16):
            mel = mel_extractor(wav)
            logit = model(mel)
        all_logits.append(logit.float().cpu())
        all_targets.append(tgt)
    all_logits = torch.cat(all_logits)
    all_targets = torch.cat(all_targets)

    from sklearn.metrics import roc_auc_score
    probs = torch.sigmoid(all_logits).numpy()
    targets_onehot = np.eye(N_CLASSES)[all_targets.numpy()]

    # Per-class AUC
    aucs_per_class = []
    class_indices_evaluated = []
    for c in range(N_CLASSES):
        if targets_onehot[:, c].sum() < 2:
            continue
        try:
            auc = roc_auc_score(targets_onehot[:, c], probs[:, c])
            aucs_per_class.append(auc)
            class_indices_evaluated.append(c)
        except Exception:
            pass
    aucs_arr = np.array(aucs_per_class)
    val_macro = float(np.mean(aucs_arr)) if len(aucs_arr) else 0.0

    # val_ns22 = average of top-22 non-saturated (AUC<1.0) species (matches exp052/053 metric)
    ns_aucs = aucs_arr[aucs_arr < 1.0]
    if len(ns_aucs) >= 22:
        val_ns22 = float(np.sort(ns_aucs)[:22].mean())  # 22 lowest (hardest) ns species
    elif len(ns_aucs) > 0:
        val_ns22 = float(ns_aucs.mean())
    else:
        val_ns22 = val_macro

    # Per-taxon AUC
    taxon_aucs = {}
    species_df_ = species_df.copy()
    species_df_["idx"] = species_df_["primary_label"].map(label_to_idx)
    for cls in species_df_["class_name"].unique():
        idx_list = species_df_[species_df_["class_name"] == cls]["idx"].tolist()
        cls_aucs = []
        for c in idx_list:
            if targets_onehot[:, c].sum() < 2:
                continue
            try: cls_aucs.append(roc_auc_score(targets_onehot[:, c], probs[:, c]))
            except: pass
        taxon_aucs[cls] = float(np.mean(cls_aucs)) if cls_aucs else float("nan")

    # Class-level stats (matches exp053 log)
    class_stats = {
        "n": len(aucs_arr),
        "median": float(np.median(aucs_arr)) if len(aucs_arr) else 0.0,
        "p25": float(np.percentile(aucs_arr, 25)) if len(aucs_arr) else 0.0,
        "p75": float(np.percentile(aucs_arr, 75)) if len(aucs_arr) else 0.0,
        "n_gt05": int((aucs_arr > 0.5).sum()),
        "n_gt07": int((aucs_arr > 0.7).sum()),
        "n_gt09": int((aucs_arr > 0.9).sum()),
        "n_perfect": int((aucs_arr == 1.0).sum()),
    }
    return val_macro, val_ns22, taxon_aucs, class_stats


In [ ]:
# Cell 13: main train loop (log format matches exp052/053)
best_val = 0.0
best_path = DRIVE_ROOT / "stage1_v2_best.pth"
log_path = DRIVE_ROOT / "stage1_v2_train.log"

step = 0
total_t0 = time.time()
for ep in range(CFG["epochs"]):
    t0 = time.time()
    model.train()
    losses = []
    for batch_i, (wav, tgt) in enumerate(train_loader):
        wav = wav.to(DEVICE, non_blocking=True)
        tgt = tgt.to(DEVICE)

        if CFG["mixup_alpha"] > 0:
            wav, tgt_a, tgt_b, lam = mixup_audio(wav, tgt, CFG["mixup_alpha"])
        else:
            tgt_a = tgt; tgt_b = tgt; lam = 1.0

        with autocast('cuda', dtype=torch.bfloat16):
            mel = mel_extractor(wav)
            mel = spec_augment(mel, freq_mask=30, time_mask=60)
            logit = model(mel)
            tgt_one_a = F.one_hot(tgt_a, N_CLASSES).float()
            tgt_one_b = F.one_hot(tgt_b, N_CLASSES).float()
            ls = CFG["label_smoothing"]
            tgt_smooth_a = tgt_one_a * (1 - ls) + ls / N_CLASSES
            tgt_smooth_b = tgt_one_b * (1 - ls) + ls / N_CLASSES
            tgt_smooth = lam * tgt_smooth_a + (1 - lam) * tgt_smooth_b
            loss = F.binary_cross_entropy_with_logits(logit, tgt_smooth)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
        optimizer.step()
        scheduler.step()
        losses.append(loss.item())
        step += 1
        if batch_i % 100 == 0:
            print(f"  [ep{ep+1} step {batch_i}/{len(train_loader)}] loss={loss.item():.4f} lr={scheduler.get_last_lr()[0]:.2e}")

    avg_loss = float(np.mean(losses))
    val_macro, val_ns22, taxon_aucs, cstat = evaluate()
    ep_elapsed = time.time() - t0
    total_elapsed = time.time() - total_t0
    cur_lr = scheduler.get_last_lr()[0]

    # Determine BEST flag
    is_best = val_ns22 > best_val
    best_mark = "BEST" if is_best else ""

    # Log line 1: header (exp052/053 style)
    log_l1 = (f"=== Ep {ep+1}/{CFG['epochs']}: loss={avg_loss:.4f} "
              f"val_ns22={val_ns22:.4f} val_macro={val_macro:.4f} {best_mark} "
              f"lr={cur_lr:.2e} "
              f"({ep_elapsed/60:.1f}min, total {total_elapsed/60:.1f}min) ===")
    # Log line 2: taxon
    log_l2 = "    taxon: " + " ".join(f"{k}={v:.3f}" if not (v != v) else f"{k}=nan"
                                       for k, v in taxon_aucs.items())
    # Log line 3: class stats
    log_l3 = (f"    class: n={cstat['n']} median={cstat['median']:.3f} "
              f"p25={cstat['p25']:.3f} p75={cstat['p75']:.3f} "
              f"#>0.5={cstat['n_gt05']} #>0.7={cstat['n_gt07']} "
              f"#>0.9={cstat['n_gt09']} #perfect={cstat['n_perfect']}")

    print(log_l1); print(log_l2); print(log_l3)
    with open(log_path, "a") as f:
        f.write(log_l1 + "\n" + log_l2 + "\n" + log_l3 + "\n")

    if is_best:
        best_val = val_ns22
        torch.save({"state_dict": model.state_dict(),
                    "val_ns22": val_ns22, "val_macro": val_macro,
                    "ep": ep+1, "cfg": CFG}, best_path)
        print(f"    BEST saved val_ns22={val_ns22:.4f}")

print(f"\nDone. Best val_ns22: {best_val:.4f}")
print(f"Best ckpt: {best_path}")


In [ ]:
# Cell 14: upload backbone weights to Kaggle Dataset
import json, os, subprocess
from pathlib import Path

USER = "maekeso"
SLUG = "birdclef2026-exp054-stage1-effv2s"
TITLE = "BirdCLEF2026 exp054 Stage 1 v2 EffNetV2-S backbone"

UPLOAD_DIR = Path("/content/upload_stage1_v2")
UPLOAD_DIR.mkdir(exist_ok=True, parents=True)
shutil.copy(best_path, UPLOAD_DIR / "stage1_v2_best.pth")
shutil.copy(log_path, UPLOAD_DIR / "stage1_v2_train.log")

meta = {"title": TITLE, "id": f"{USER}/{SLUG}", "licenses": [{"name": "other"}]}
(UPLOAD_DIR / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))

from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
try:
    api.dataset_create_new(folder=str(UPLOAD_DIR), public=False, dir_mode="zip", quiet=False)
    print(f"OK new dataset")
except Exception as e:
    print(f"create_new err: {str(e)[:200]}")
    try:
        api.dataset_create_version(folder=str(UPLOAD_DIR), version_notes="stage1 v2",
                                    dir_mode="zip", quiet=False)
        print("OK version created")
    except Exception as e2:
        print(f"create_version err: {str(e2)[:200]}")
print(f"\nURL: https://www.kaggle.com/datasets/{USER}/{SLUG}")


In [ ]:
# Cell 15: summary
print(f"=== Stage 1 v2 complete ===")
print(f"Best val_macro: {best_val:.4f}")
print(f"Total epochs: {CFG['epochs']}")
print(f"Dataset uploaded: maekeso/{SLUG}")
print(f"\nNext: Stage 2A v2 (BC2026 finetune) with this backbone")


In [ ]:
# Cell 16: auto-disconnect (save compute units)
from google.colab import runtime
runtime.unassign()
